# 🚦 Project 6 — GuardGate: Human-in-the-Loop Approval Agent

**Core Concept:** Confidence-based escalation with human approval workflow

### Architecture
Agent Decision
      │
      ▼
Confidence Check
      │
High → Auto Execute
Low  → Human Approval Required
      │
      ▼
Audit Log → Result

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.4 MB/s eta 0:00:00


API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "Key"

All Setup In One Block

In [3]:
import os
import json
import time
from datetime import datetime, timezone
from enum import Enum
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Enums ────────────────────────────────────────────────────
class DecisionStatus(Enum):
    AUTO_APPROVED = "auto_approved"
    PENDING_APPROVAL = "pending_approval"
    HUMAN_APPROVED = "human_approved"
    HUMAN_REJECTED = "human_rejected"
    AUTO_REJECTED = "auto_rejected"

class RiskLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

# ── Risk Assessor ────────────────────────────────────────────
class RiskAssessor:
    def __init__(self):
        self.risk_rules = {
            "financial": {
                "keywords": ["refund", "payment", "transfer", "charge", "invoice", "money"],
                "base_risk": 0.6
            },
            "data": {
                "keywords": ["delete", "remove", "drop", "purge", "wipe"],
                "base_risk": 0.7
            },
            "access": {
                "keywords": ["admin", "permission", "role", "access", "privilege"],
                "base_risk": 0.5
            },
            "communication": {
                "keywords": ["email", "send", "notify", "broadcast", "announce"],
                "base_risk": 0.3
            },
            "legal": {
                "keywords": ["contract", "agreement", "sign", "legal", "compliance"],
                "base_risk": 0.8
            }
        }

    def assess(self, action: str, amount: float = 0) -> dict:
        action_lower = action.lower()
        max_risk = 0.1
        risk_category = "general"

        for category, rules in self.risk_rules.items():
            for keyword in rules["keywords"]:
                if keyword in action_lower:
                    risk = rules["base_risk"]
                    if amount > 10000:
                        risk = min(1.0, risk + 0.2)
                    elif amount > 1000:
                        risk = min(1.0, risk + 0.1)
                    if risk > max_risk:
                        max_risk = risk
                        risk_category = category

        if max_risk < 0.3:
            risk_level = RiskLevel.LOW
        elif max_risk < 0.5:
            risk_level = RiskLevel.MEDIUM
        elif max_risk < 0.75:
            risk_level = RiskLevel.HIGH
        else:
            risk_level = RiskLevel.CRITICAL

        return {
            "risk_score": round(max_risk, 2),
            "risk_level": risk_level.value,
            "risk_category": risk_category,
            "confidence": round(1 - max_risk, 2)
        }

# ── Audit Logger ─────────────────────────────────────────────
class AuditLogger:
    def __init__(self):
        self.logs = []

    def log(self, action: str, status: str, risk_score: float,
            human_reviewer: str = None, notes: str = None):
        entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "action": action,
            "status": status,
            "risk_score": risk_score,
            "human_reviewer": human_reviewer,
            "notes": notes
        }
        self.logs.append(entry)
        logger.info(f"Audit: {action[:40]} | Status: {status} | Risk: {risk_score}")

    def get_report(self) -> dict:
        total = len(self.logs)
        auto_approved = sum(1 for l in self.logs if l["status"] == "auto_approved")
        human_approved = sum(1 for l in self.logs if l["status"] == "human_approved")
        rejected = sum(1 for l in self.logs if "rejected" in l["status"])

        return {
            "total_decisions": total,
            "auto_approved": auto_approved,
            "human_approved": human_approved,
            "rejected": rejected,
            "logs": self.logs
        }

# ── Human Approval Simulator ─────────────────────────────────
class HumanApprovalSimulator:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a human reviewer for an AI approval system.
You review AI decisions that require human oversight.

Evaluate the action carefully and decide:
- APPROVE if the action seems reasonable and the risk is manageable
- REJECT if the action seems too risky or inappropriate

Respond ONLY in this exact format:
DECISION: APPROVE or REJECT
REASON: [one line explanation]
CONFIDENCE: [your confidence 0.0 to 1.0]"""),
            ("human", """Action requiring approval:
Action: {action}
Risk Score: {risk_score}
Risk Level: {risk_level}
Risk Category: {risk_category}
Amount: {amount}

Please review and decide:""")
        ])

    def review(self, action: str, risk_assessment: dict, amount: float = 0) -> dict:
        logger.info(f"Simulating human review for: {action[:50]}")
        chain = self.prompt | self.llm
        response = chain.invoke({
            "action": action,
            "risk_score": risk_assessment["risk_score"],
            "risk_level": risk_assessment["risk_level"],
            "risk_category": risk_assessment["risk_category"],
            "amount": amount
        })

        raw = response.content
        logger.debug(f"Human reviewer response: {raw}")

        decision = "REJECT"
        reason = "Could not parse decision"
        confidence = 0.5

        for line in raw.strip().split('\n'):
            if line.startswith("DECISION:"):
                decision = line.replace("DECISION:", "").strip()
            elif line.startswith("REASON:"):
                reason = line.replace("REASON:", "").strip()
            elif line.startswith("CONFIDENCE:"):
                try:
                    confidence = float(line.replace("CONFIDENCE:", "").strip())
                except:
                    confidence = 0.5

        return {
            "decision": decision,
            "reason": reason,
            "confidence": confidence,
            "reviewer": "human_simulator"
        }

# ── GuardGate Agent ──────────────────────────────────────────
class GuardGateAgent:
    def __init__(self, auto_approve_threshold: float = 0.7):
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.1,
            api_key=os.environ["GROQ_API_KEY"]
        )
        self.auto_approve_threshold = auto_approve_threshold
        self.risk_assessor = RiskAssessor()
        self.audit_logger = AuditLogger()
        self.human_reviewer = HumanApprovalSimulator(self.llm)

        self.analysis_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are GuardGate, an AI decision analysis agent.
Analyze the requested action and provide a structured assessment.

Respond in this exact format:
ACTION_SUMMARY: [one line summary of what will happen]
IMPACT: [what the impact of this action will be]
RECOMMENDATION: PROCEED or CAUTION or BLOCK"""),
            ("human", "Analyze this action: {action}")
        ])

        logger.info(f"GuardGate initialized | Auto-approve threshold: {auto_approve_threshold}")

    def process(self, action: str, amount: float = 0,
                requested_by: str = "system") -> dict:
        logger.info(f"Processing: {action[:50]} | Amount: ${amount}")

        chain = self.analysis_prompt | self.llm
        analysis_response = chain.invoke({"action": action})
        analysis = analysis_response.content

        action_summary = ""
        impact = ""
        recommendation = ""
        for line in analysis.strip().split('\n'):
            if line.startswith("ACTION_SUMMARY:"):
                action_summary = line.replace("ACTION_SUMMARY:", "").strip()
            elif line.startswith("IMPACT:"):
                impact = line.replace("IMPACT:", "").strip()
            elif line.startswith("RECOMMENDATION:"):
                recommendation = line.replace("RECOMMENDATION:", "").strip()

        risk = self.risk_assessor.assess(action, amount)
        confidence = risk["confidence"]

        result = {
            "action": action,
            "action_summary": action_summary,
            "impact": impact,
            "recommendation": recommendation,
            "risk_score": risk["risk_score"],
            "risk_level": risk["risk_level"],
            "risk_category": risk["risk_category"],
            "confidence": confidence,
            "amount": amount,
            "requested_by": requested_by
        }

        if confidence >= self.auto_approve_threshold:
            result["status"] = DecisionStatus.AUTO_APPROVED.value
            result["message"] = "Action auto-approved — confidence above threshold"
            self.audit_logger.log(action, "auto_approved", risk["risk_score"])
            logger.info(f"AUTO APPROVED | Confidence: {confidence}")

        elif recommendation == "BLOCK":
            result["status"] = DecisionStatus.AUTO_REJECTED.value
            result["message"] = "Action blocked — AI recommendation is BLOCK"
            self.audit_logger.log(action, "auto_rejected", risk["risk_score"])
            logger.warning(f"AUTO REJECTED | Recommendation: BLOCK")

        else:
            logger.warning(f"ESCALATING TO HUMAN | Confidence: {confidence}")
            result["status"] = DecisionStatus.PENDING_APPROVAL.value
            result["message"] = "Low confidence — escalating to human reviewer"

            human_decision = self.human_reviewer.review(action, risk, amount)
            result["human_review"] = human_decision

            if human_decision["decision"] == "APPROVE":
                result["status"] = DecisionStatus.HUMAN_APPROVED.value
                result["message"] = f"Human approved: {human_decision['reason']}"
                self.audit_logger.log(
                    action, "human_approved",
                    risk["risk_score"],
                    human_decision["reviewer"],
                    human_decision["reason"]
                )
                logger.info(f"HUMAN APPROVED | Reason: {human_decision['reason']}")
            else:
                result["status"] = DecisionStatus.HUMAN_REJECTED.value
                result["message"] = f"Human rejected: {human_decision['reason']}"
                self.audit_logger.log(
                    action, "human_rejected",
                    risk["risk_score"],
                    human_decision["reviewer"],
                    human_decision["reason"]
                )
                logger.warning(f"HUMAN REJECTED | Reason: {human_decision['reason']}")

        return result

    def display_result(self, result: dict):
        print("\n" + "="*55)
        print("GUARDGATE DECISION")
        print("="*55)
        print(f"Action      : {result['action']}")
        print(f"Summary     : {result.get('action_summary', 'N/A')}")
        print(f"Risk Level  : {result['risk_level'].upper()}")
        print(f"Risk Score  : {result['risk_score']}")
        print(f"Confidence  : {result['confidence']}")
        print(f"Status      : {result['status'].upper()}")
        print(f"Message     : {result['message']}")
        if "human_review" in result:
            hr = result["human_review"]
            print(f"\nHuman Review:")
            print(f"  Decision  : {hr['decision']}")
            print(f"  Reason    : {hr['reason']}")
        print("="*55)

agent = GuardGateAgent(auto_approve_threshold=0.7)
print("GuardGate agent ready")

04:31:57 | INFO | GuardGate initialized | Auto-approve threshold: 0.7
GuardGate agent ready


Test High Risk Actions

In [4]:
print("========== HIGH RISK ACTIONS ==========\n")

high_risk_actions = [
    ("Process refund of $5000 for customer order #12345", 5000),
    ("Delete all user records from the staging database", 0),
    ("Sign contract agreement with vendor for $50000", 50000)
]

for action, amount in high_risk_actions:
    result = agent.process(action, amount, requested_by="admin")
    agent.display_result(result)

========== HIGH RISK ACTIONS ==========

04:31:57 | INFO | Processing: Process refund of $5000 for customer order #12345 | Amount: $5000
04:31:58 | WARNING | ESCALATING TO HUMAN | Confidence: 0.3
04:31:58 | INFO | Simulating human review for: Process refund of $5000 for customer order #12345
04:31:58 | DEBUG | Human reviewer response: DECISION: REJECT
REASON: The high risk score and large refund amount of $5000 warrant further investigation to ensure the refund is legitimate and not fraudulent.
CONFIDENCE: 0.8
04:31:58 | INFO | Audit: Process refund of $5000 for customer ord | Status: human_rejected | Risk: 0.7
04:31:58 | WARNING | HUMAN REJECTED | Reason: The high risk score and large refund amount of $5000 warrant further investigation to ensure the refund is legitimate and not fraudulent.

GUARDGATE DECISION
Action      : Process refund of $5000 for customer order #12345
Summary     : A refund of $5000 will be processed for customer order #12345.
Risk Level  : HIGH
Risk Score  : 0.7

Test Critical Risk

In [5]:
print("========== CRITICAL RISK ACTION ==========\n")

result = agent.process(
    "Transfer $100000 to external vendor account immediately",
    amount=100000,
    requested_by="automated_system"
)
agent.display_result(result)

========== CRITICAL RISK ACTION ==========

04:32:00 | INFO | Processing: Transfer $100000 to external vendor account immedi | Amount: $100000
04:32:00 | WARNING | ESCALATING TO HUMAN | Confidence: 0.2
04:32:00 | INFO | Simulating human review for: Transfer $100000 to external vendor account immedi
04:32:00 | DEBUG | Human reviewer response: DECISION: REJECT
REASON: The high risk score and critical risk level indicate a potentially unsafe large financial transaction.
CONFIDENCE: 0.9
04:32:00 | INFO | Audit: Transfer $100000 to external vendor acco | Status: human_rejected | Risk: 0.8
04:32:00 | WARNING | HUMAN REJECTED | Reason: The high risk score and critical risk level indicate a potentially unsafe large financial transaction.

GUARDGATE DECISION
Action      : Transfer $100000 to external vendor account immediately
Summary     : Initiating an immediate transfer of $100,000 to an external vendor account.
Risk Level  : CRITICAL
Risk Score  : 0.8
Confidence  : 0.2
Status      : HUMAN_R

Audit Report

In [6]:
print("========== AUDIT REPORT ==========\n")
report = agent.audit_logger.get_report()
print(f"Total Decisions : {report['total_decisions']}")
print(f"Auto Approved   : {report['auto_approved']}")
print(f"Human Approved  : {report['human_approved']}")
print(f"Rejected        : {report['rejected']}")
print(f"\nDetailed Audit Log:")
for log in report['logs']:
    print(f"  [{log['timestamp'][:19]}] {log['action'][:40]:40} | {log['status']:20} | Risk: {log['risk_score']}")

========== AUDIT REPORT ==========

Total Decisions : 4
Auto Approved   : 0
Human Approved  : 0
Rejected        : 4

Detailed Audit Log:
  [2026-08-09T04:31:58] Process refund of $5000 for customer ord | human_rejected       | Risk: 0.7
  [2026-08-09T04:31:59] Delete all user records from the staging | human_rejected       | Risk: 0.7
  [2026-08-09T04:32:00] Sign contract agreement with vendor for  | human_rejected       | Risk: 1.0
  [2026-08-09T04:32:00] Transfer $100000 to external vendor acco | human_rejected       | Risk: 0.8


Project Summary

In [7]:
print("========== GUARDGATE SUMMARY ==========\n")
print("Project      : GuardGate — Human-in-the-Loop Approval Agent")
print("Author       : K Murali Krishna")
print("Model        : Groq LLaMA-3.3-70b-versatile")
print("\nDecision Flow:")
print("  Confidence >= 0.7 → Auto Approved")
print("  Confidence < 0.7  → Human Review Required")
print("  AI says BLOCK     → Auto Rejected")
print("\nKey Capabilities:")
print("  ✓ Risk scoring based on action type and amount")
print("  ✓ Confidence-based auto approval threshold")
print("  ✓ Human-in-the-loop escalation")
print("  ✓ Simulated human reviewer using LLM")
print("  ✓ Complete audit trail for every decision")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Human-in-the-loop pattern")
print("  ✓ Risk assessment engine")
print("  ✓ Audit logging for compliance")
print("  ✓ Threshold-based decision routing")

========== GUARDGATE SUMMARY ==========

Project      : GuardGate — Human-in-the-Loop Approval Agent
Author       : K Murali Krishna
Model        : Groq LLaMA-3.3-70b-versatile

Decision Flow:
  Confidence >= 0.7 → Auto Approved
  Confidence < 0.7  → Human Review Required
  AI says BLOCK     → Auto Rejected

Key Capabilities:
  ✓ Risk scoring based on action type and amount
  ✓ Confidence-based auto approval threshold
  ✓ Human-in-the-loop escalation
  ✓ Simulated human reviewer using LLM
  ✓ Complete audit trail for every decision

Production Concepts Demonstrated:
  ✓ Human-in-the-loop pattern
  ✓ Risk assessment engine
  ✓ Audit logging for compliance
  ✓ Threshold-based decision routing
